To get some safe temporal numeric offsets for v1 

In [3]:
import pandas as pd

df = pd.read_csv("../../data/processed/training_level1_full.csv")
print(df.shape)
print(df.columns.tolist())

(2082, 132)
['player_id', 'season', 'salary_usd', 'log_salary', 'salary_cap', 'salary_cap_ratio', 'log_salary_cap_ratio', 'salary_cap_equiv', 'Age', 'GP', 'W', 'L', 'Min', 'PTS', 'FGM', 'FGA', 'FG%', '3PM', '3PA', '3P%', 'FTM', '_is_lottery', 'draft_is_first_round', 'draft_is_second_round', 'undrafted_flag', 'agent_name', 'agent_client_count', 'agent_name_all', 'agent_is_top', 'age_now', 'years_since_draft', 'team_value_usd', 'team_value_pct', 'team_big_market_flag', 'city', 'state', 'award_All-Defensive', 'award_All-NBA', 'award_All-Rookie', 'award_All-Star MVP', 'award_Clutch Player of the Year', 'award_Coach of tFTA', 'FT%', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'PF', 'FP', 'DD2', 'TD3', '+/-', 'PTS_per_min', 'REB_per_min', 'AST_per_min', 'STL_per_min', 'BLK_per_min', 'TOV_per_min', 'FGA_per_min', 'FGM_per_min', '3PA_per_min', '3PM_per_min', 'FTA_per_min', 'FTM_per_min', 'OREB_per_min', 'DREB_per_min', 'PF_per_min', 'PTS_per_gp', 'REB_per_gp', 'AST_per_gp', 'STL_per_gp'

In [4]:
time_like_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in ["age", "year", "season", "draft", "experience"])
]

time_like_cols

['season',
 'Age',
 'draft_is_first_round',
 'draft_is_second_round',
 'undrafted_flag',
 'agent_name',
 'agent_client_count',
 'agent_name_all',
 'agent_is_top',
 'age_now',
 'years_since_draft',
 'award_Clutch Player of the Year',
 'draft_year',
 'drafthe Year',
 'award_Defensive Player of the Year',
 'award_Rookie of the Year',
 'award_Sixth Man of the Year',
 'award_Twyman-Stokes Teammate of the Year Award']

发现只有age_now 和years_since_draft 可以作为v1的额外特征 接下来为这两个特征跑R^2 看看是不是可以单独解释薪资很多

In [6]:
cols = ["age_now", "years_since_draft"]
# 只用训练期，防止任何时间泄露
train = df[df["season"] < 2024].copy()
for c in cols:
    if c in train.columns:
        print(
            c,
            "missing_ratio =",
            train[c].isna().mean()
        )
    else:
        print(c, "NOT FOUND")

age_now missing_ratio = 0.0
years_since_draft missing_ratio = 0.0


In [7]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import numpy as np

def single_feature_r2(col):
    sub = train[[col, "log_salary"]].dropna()
    
    # 样本太少就不算
    if len(sub) < 100:
        return None
    
    X = sub[[col]].values
    y = sub["log_salary"].values
    
    model = Ridge(alpha=1.0)
    model.fit(X, y)
    
    y_pred = model.predict(X)
    return r2_score(y, y_pred)

In [8]:
for c in ["age_now", "years_since_draft"]:
    r2 = single_feature_r2(c)
    print(f"{c:20s}  single-variable R² = {r2:.3f}")


age_now               single-variable R² = 0.061
years_since_draft     single-variable R² = 0.147
